In [47]:
import torch
import torch.nn.functional as F

@torch.no_grad()
def pinvconv_ls(K, y, k2, rcond=1e-5, center_weight=1.):
    Cout, Cin, kh, kw = K.shape
    sup = kh + k2 - 1
    center = sup // 2
    Kp = F.pad(K, (k2 - 1,) * 4)

    rows = []
    for c in range(Cin):
        oc = []
        for o in range(Cout):
            P = F.unfold(Kp[o, c][None, None], kernel_size=(k2, k2), stride=1)
            oc.append(P.squeeze(0).t())  # [sup*sup, k2*k2]
        rows.append(torch.cat(oc, dim=1))
    A = torch.cat(rows, dim=0)  # [Cin*sup*sup, Cout*k2*k2]

    b_all = torch.zeros(Cin, Cin * sup * sup, dtype=K.dtype, device=K.device)
    center_row = center * sup + center
    for s in range(Cin):
        b_all[s, s * (sup * sup) + center_row] = 1.0

    w = torch.ones(Cin * sup * sup, dtype=A.dtype, device=A.device)
    for s in range(Cin):
        idx = s * (sup * sup) + center_row
        w[idx] = center_weight       
    Aw = A * w[:, None]

    G = torch.empty(Cin, Cout, k2, k2, dtype=K.dtype, device=K.device)
    for s in range(Cin):
        bw = b_all[s] * w

        g = torch.linalg.lstsq(Aw, bw, rcond=rcond).solution   # [Cout*k2*k2]
        Gi = g.view(Cout, k2, k2)
        Gi = torch.flip(Gi, dims=(-2, -1)) 
        G[s] = Gi

    x_hat = F.conv2d(y, G, padding=k2 // 2)
    return x_hat


# ---------- quick test ----------
B, Cin, Cout, H, W = 2, 3, 3, 64, 64
K_pinv_sz = 35
K_sz = 5

# random input and kernel
x = torch.ones(B, Cin, H, W)
x =F.pad(x, (32,)*4)
# K = torch.zeros(Cout, Cin, K_sz, K_sz)
# K[:, :, 2, 2] = torch.eye(3)
K = torch.randn(Cout, Cin, K_sz, K_sz) 
# forward "same" conv
y = F.conv2d(x, K, padding=K_sz//2)

# invert with LS (small kernel)
x_hat = pinvconv_ls(K, y, K_pinv_sz)

# report
rel_err = (x_hat - x).pow(2).sum().sqrt() / (x.pow(2).sum().sqrt() + 1e-12)
print(f"Relative RMSE: {rel_err.item():.6f}")


Relative RMSE: 0.056442


In [25]:
K = torch.randn(Cout, Cin, K_sz, K_sz) + 1